# Benchmark: WDM Time-Shift Backends

This notebook compares lag-first chunked backends against legacy assembly on laptop-safe sizes.
Run using the `lisatools` conda environment. vmap is intentionally disabled by default.

In [1]:
# --- Cell 1: imports and environment ---

import os
import sys
import csv
import time
from datetime import datetime

import numpy as np

import jax
import jax.numpy as jnp

from few.utils.constants import YRSID_SI
from few.waveform import GenerateEMRIWaveform


from lisaorbits.file import read_orbit_file
from lisaorbits import InterpolatedOrbits

# Adjust these imports only if your local package paths differ.
from WDM_LISAresponse.code.lisa_response.waveforms import generate_windowed_padded_waveform
from WDM_LISAresponse.code.lisa_response.response import (
    one_arm_response_td,
    one_arm_response_wdm_dT,
)
from WDM_LISAresponse.code.lisa_response.TDI import (
    gen2_michelson_wdm,
    gen2_michelson_td,
)

from WDM.code.time_delay_filters import time_shift_fast as tsf
import WDM

# Context-reused one-arm benchmarking helpers
from WDM_LISAresponse.code.lisa_response.response_sub import build_sky_basis
from WDM_LISAresponse.code.lisa_response.response import (
    _build_wdm_dT_context,
    _one_arm_response_wdm_dT_from_context,
)

print("Python executable:", sys.executable)
print("Python version:", sys.version)
#print("JAX devices:", jax.devices())
print("Current working directory:", os.getcwd())
print("Run this notebook from the lisatools conda environment.")

Python executable: /data/tk593/conda-envs/lisatools/bin/python
Python version: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:16:04) [GCC 11.2.0]
Current working directory: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples
Run this notebook from the lisatools conda environment.


In [14]:
# --- Cell 2: backend configurations ---

# -------------------------------------------------------------------
# Dense lag-support candidates
# -------------------------------------------------------------------

L_TRUNC_REF = 25

# Reference/safe/aggressive dense comparison values.
L_TRUNC_SWEEP = [16, 20, 25]


# -------------------------------------------------------------------
# Reference / safe / aggressive dense settings
# -------------------------------------------------------------------

REFERENCE_DENSE_L_TRUNC = 25
REFERENCE_DENSE_SHIFT_KWARGS = {
    "assembly_backend": "lagfirst_chunked_lagblock",
    "assembly_precision": "complex64",
    "row_chunk_size": 2048,
    "lag_block_size": 32,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
}

# Final safe dense default.
# This is the recommended dense setting after lag/lag-block tuning.
SAFE_FAST_DENSE_L_TRUNC = 20
SAFE_FAST_DENSE_SHIFT_KWARGS = {
    "assembly_backend": "lagfirst_chunked_lagblock",
    "assembly_precision": "complex64",
    "row_chunk_size": 2048,
    "lag_block_size": 16,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
}

# Aggressive dense option.
# Slightly faster than L20, but mismatch relative to L25 is around 1e-7 scale.
AGGRESSIVE_FAST_DENSE_L_TRUNC = 16
AGGRESSIVE_FAST_DENSE_SHIFT_KWARGS = {
    "assembly_backend": "lagfirst_chunked_lagblock",
    "assembly_precision": "complex64",
    "row_chunk_size": 2048,
    "lag_block_size": 16,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
}

# Baseline non-lagblock backend, useful as a sanity comparison.
BASELINE_CHUNKED_KWARGS = {
    "assembly_backend": "lagfirst_chunked",
    "assembly_precision": "complex64",
    "row_chunk_size": 1024,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
}


# -------------------------------------------------------------------
# Final recommended dense kwargs
# -------------------------------------------------------------------

BEST_DENSE_ONE_ARM_KWARGS = dict(SAFE_FAST_DENSE_SHIFT_KWARGS)

BEST_DENSE_TDI_L_TRUNC = SAFE_FAST_DENSE_L_TRUNC
BEST_DENSE_TDI_KWARGS = {
    **dict(SAFE_FAST_DENSE_SHIFT_KWARGS),
    "batch_chunk": 1,
}
BEST_DENSE_TDI_ASSEMBLY_MODE = "batch_map"

AGGRESSIVE_DENSE_ONE_ARM_KWARGS = dict(AGGRESSIVE_FAST_DENSE_SHIFT_KWARGS)

AGGRESSIVE_DENSE_TDI_L_TRUNC = AGGRESSIVE_FAST_DENSE_L_TRUNC
AGGRESSIVE_DENSE_TDI_KWARGS = {
    **dict(AGGRESSIVE_FAST_DENSE_SHIFT_KWARGS),
    "batch_chunk": 1,
}
AGGRESSIVE_DENSE_TDI_ASSEMBLY_MODE = "batch_map"


# -------------------------------------------------------------------
# Experimental modes kept for optional diagnostics only
# -------------------------------------------------------------------

# Accumulator is numerically correct and useful structurally, but not faster
# for the tuned dense path.
EXPERIMENTAL_ACCUMULATOR_TDI_ASSEMBLY_MODE = "accumulate"
EXPERIMENTAL_ACCUMULATOR_CHUNK = 1

# Job-block backend is numerically valid, but slower on CPU for dense runs.
EXPERIMENTAL_JOBBLOCK_BACKEND = "lagfirst_chunked_lagblock_jobblock"


# -------------------------------------------------------------------
# Retune helpers
# -------------------------------------------------------------------

ROW_CHUNK_SIZES = [1024, 2048, 4096]
INTERP_POINTS_SWEEP = [24, 32]


def n_lags_from_L(L_trunc):
    return 2 * int(L_trunc) + 1


def lag_blocks_for_L(L_trunc):
    """
    Candidate lag-block sizes for retuning.

    Full n_lags blocks such as 33, 41, 51 were consistently slow, so
    we keep only friendly block sizes by default.
    """
    n_lags = n_lags_from_L(L_trunc)
    candidates = [8, 16, 24, 32]
    return sorted({int(b) for b in candidates if 1 <= int(b) <= n_lags})


def kwargs_for_L_trunc(
    L_trunc,
    *,
    row_chunk_size=2048,
    interp_points=32,
    lag_block_size=None,
):
    """
    Build a lag-block backend configuration for a given L_trunc.
    """
    kwargs = dict(REFERENCE_DENSE_SHIFT_KWARGS)
    kwargs["row_chunk_size"] = int(row_chunk_size)
    kwargs["tl_tp_interp_points"] = int(interp_points)

    n_lags = n_lags_from_L(L_trunc)

    if lag_block_size is None:
        # Defaults chosen from focused retune:
        # L16/L20 prefer B16; L25 reference uses B32.
        if int(L_trunc) in (16, 20):
            lag_block_size = 16
        else:
            lag_block_size = 32

    kwargs["lag_block_size"] = int(min(lag_block_size, n_lags))
    return kwargs


# -------------------------------------------------------------------
# Switches for one-arm/backend cells
# -------------------------------------------------------------------

RUN_BASELINE = True
RUN_LTRUNC_SWEEP = True
RUN_RETUNE_GRID = False
RUN_TDI = False


def selected_backends():
    """
    Backend configurations for one-arm/backend benchmark cells.

    L_trunc is kept outside kwargs because it is passed separately to
    wdm_time_shift_variable(..., L_trunc=L).
    """
    out = {}

    if RUN_BASELINE:
        out["baseline_chunked_R1024_L25"] = {
            "L_trunc": L_TRUNC_REF,
            "kwargs": dict(BASELINE_CHUNKED_KWARGS),
        }

    if RUN_LTRUNC_SWEEP:
        out["lagblock_L16_R2048_B16"] = {
            "L_trunc": AGGRESSIVE_FAST_DENSE_L_TRUNC,
            "kwargs": dict(AGGRESSIVE_FAST_DENSE_SHIFT_KWARGS),
        }
        out["lagblock_L20_R2048_B16"] = {
            "L_trunc": SAFE_FAST_DENSE_L_TRUNC,
            "kwargs": dict(SAFE_FAST_DENSE_SHIFT_KWARGS),
        }
        out["lagblock_L25_R2048_B32"] = {
            "L_trunc": REFERENCE_DENSE_L_TRUNC,
            "kwargs": dict(REFERENCE_DENSE_SHIFT_KWARGS),
        }

    if RUN_RETUNE_GRID:
        PROMISING_L_TRUNC = [16, 20, 25]

        for L in PROMISING_L_TRUNC:
            for R in ROW_CHUNK_SIZES:
                for B in lag_blocks_for_L(L):
                    for ip in INTERP_POINTS_SWEEP:
                        kwargs = kwargs_for_L_trunc(
                            L,
                            row_chunk_size=R,
                            interp_points=ip,
                            lag_block_size=B,
                        )
                        out[f"retune_L{L}_R{R}_B{B}_ip{ip}"] = {
                            "L_trunc": int(L),
                            "kwargs": kwargs,
                        }

    return out


# -------------------------------------------------------------------
# TDI candidate construction
# -------------------------------------------------------------------

def make_tdi_candidate(
    *,
    L_trunc,
    shift_kwargs,
    tdi_assembly_mode="batch_map",
    accumulator_chunk=1,
    batch_chunk=1,
):
    """
    Build a TDI benchmark candidate.

    one_arm_kwargs are used for A1 six-link construction.
    downstream_kwargs are used for B-stage downstream TDI shifts.
    """
    one_arm_kwargs = dict(shift_kwargs)

    downstream_kwargs = dict(shift_kwargs)
    downstream_kwargs["batch_chunk"] = int(batch_chunk)

    return {
        "L_trunc": int(L_trunc),
        "one_arm_kwargs": one_arm_kwargs,
        "downstream_kwargs": downstream_kwargs,
        "tdi_assembly_mode": str(tdi_assembly_mode),
        "accumulator_chunk": int(accumulator_chunk),
    }


# -------------------------------------------------------------------
# Final focused TDI candidates
# -------------------------------------------------------------------

TDI_CANDIDATES = {}

# L25 reference.
TDI_CANDIDATES["tdi_ref_L25_batch_map"] = make_tdi_candidate(
    L_trunc=REFERENCE_DENSE_L_TRUNC,
    shift_kwargs=REFERENCE_DENSE_SHIFT_KWARGS,
    tdi_assembly_mode="batch_map",
    accumulator_chunk=1,
    batch_chunk=1,
)

# Recommended safe dense default.
TDI_CANDIDATES["tdi_safe_L20_batch_map"] = make_tdi_candidate(
    L_trunc=SAFE_FAST_DENSE_L_TRUNC,
    shift_kwargs=SAFE_FAST_DENSE_SHIFT_KWARGS,
    tdi_assembly_mode="batch_map",
    accumulator_chunk=1,
    batch_chunk=1,
)

# Aggressive dense option.
TDI_CANDIDATES["tdi_aggr_L16_batch_map"] = make_tdi_candidate(
    L_trunc=AGGRESSIVE_FAST_DENSE_L_TRUNC,
    shift_kwargs=AGGRESSIVE_FAST_DENSE_SHIFT_KWARGS,
    tdi_assembly_mode="batch_map",
    accumulator_chunk=1,
    batch_chunk=1,
)


# -------------------------------------------------------------------
# Optional diagnostic candidates
# -------------------------------------------------------------------
# Uncomment only when explicitly profiling/testing these paths.

# Accumulator parity check for safe default:
# TDI_CANDIDATES["tdi_safe_L20_accum_c1"] = make_tdi_candidate(
#     L_trunc=SAFE_FAST_DENSE_L_TRUNC,
#     shift_kwargs=SAFE_FAST_DENSE_SHIFT_KWARGS,
#     tdi_assembly_mode=EXPERIMENTAL_ACCUMULATOR_TDI_ASSEMBLY_MODE,
#     accumulator_chunk=EXPERIMENTAL_ACCUMULATOR_CHUNK,
#     batch_chunk=1,
# )

# Job-block diagnostic for safe default:
# jobblock_L20_J2 = {
#     **dict(SAFE_FAST_DENSE_SHIFT_KWARGS),
#     "assembly_backend": EXPERIMENTAL_JOBBLOCK_BACKEND,
#     "job_block_size": 2,
# }
# TDI_CANDIDATES["tdi_safe_L20_jobblock_J2"] = make_tdi_candidate(
#     L_trunc=SAFE_FAST_DENSE_L_TRUNC,
#     shift_kwargs=jobblock_L20_J2,
#     tdi_assembly_mode="batch_map",
#     accumulator_chunk=1,
#     batch_chunk=1,
# )


# -------------------------------------------------------------------
# Materialize backend configs
# -------------------------------------------------------------------

BACKENDS = selected_backends()


# -------------------------------------------------------------------
# Print summary
# -------------------------------------------------------------------

print("Active one-arm/backend configurations:")
for name, cfg in BACKENDS.items():
    print(f"  {name}: L_trunc={cfg['L_trunc']}, kwargs={cfg['kwargs']}")

print("\nTDI candidates:")
for name, cfg in TDI_CANDIDATES.items():
    print(f"  {name}:")
    print(f"    L_trunc: {cfg['L_trunc']}")
    print(f"    tdi_assembly_mode: {cfg['tdi_assembly_mode']}")
    print(f"    accumulator_chunk: {cfg['accumulator_chunk']}")
    print(f"    one_arm_kwargs: {cfg['one_arm_kwargs']}")
    print(f"    downstream_kwargs: {cfg['downstream_kwargs']}")

print()
print("Recommended dense one-arm kwargs:")
print(BEST_DENSE_ONE_ARM_KWARGS)

print("\nRecommended dense TDI:")
print("  L_trunc:", BEST_DENSE_TDI_L_TRUNC)
print("  kwargs:", BEST_DENSE_TDI_KWARGS)
print("  tdi_assembly_mode:", BEST_DENSE_TDI_ASSEMBLY_MODE)

print("\nAggressive dense TDI:")
print("  L_trunc:", AGGRESSIVE_DENSE_TDI_L_TRUNC)
print("  kwargs:", AGGRESSIVE_DENSE_TDI_KWARGS)
print("  tdi_assembly_mode:", AGGRESSIVE_DENSE_TDI_ASSEMBLY_MODE)

print()
print("L_TRUNC_REF:", L_TRUNC_REF)
print("L_TRUNC_SWEEP:", L_TRUNC_SWEEP)
print("ROW_CHUNK_SIZES:", ROW_CHUNK_SIZES)
print("INTERP_POINTS_SWEEP:", INTERP_POINTS_SWEEP)
print("RUN_BASELINE:", RUN_BASELINE)
print("RUN_LTRUNC_SWEEP:", RUN_LTRUNC_SWEEP)
print("RUN_RETUNE_GRID:", RUN_RETUNE_GRID)
print("RUN_TDI:", RUN_TDI)

Active one-arm/backend configurations:
  baseline_chunked_R1024_L25: L_trunc=25, kwargs={'assembly_backend': 'lagfirst_chunked', 'assembly_precision': 'complex64', 'row_chunk_size': 1024, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear'}
  lagblock_L16_R2048_B16: L_trunc=16, kwargs={'assembly_backend': 'lagfirst_chunked_lagblock', 'assembly_precision': 'complex64', 'row_chunk_size': 2048, 'lag_block_size': 16, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear'}
  lagblock_L20_R2048_B16: L_trunc=20, kwargs={'assembly_backend': 'lagfirst_chunked_lagblock', 'assembly_precision': 'complex64', 'row_chunk_size': 2048, 'lag_block_size': 16, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear'}
  lagblock_L25_R2048_B32: L_trunc=25, kwargs={'assembly_backend': 'lagfirst_chunked_lagblock', 'assembly_precision': 'complex64', 'row_chunk_size': 2048, 'lag_block_size': 32, 'tl_tp_mode': 'interp', 'tl_tp_interp_

In [6]:
# --- Cell 3: comparison and timing helpers ---

from WDM_LISAresponse.code.utils.utils import (
    wdm_mismatch,
    compare_response_signals,
    compute_wdm_cut_info,
    apply_wdm_cut,
)


def rel_l2(test, ref):
    test = np.asarray(test)
    ref = np.asarray(ref)
    num = np.linalg.norm(test - ref)
    den = np.linalg.norm(ref)
    return float(num / den) if den != 0 else np.nan


def max_abs(test, ref):
    test = np.asarray(test)
    ref = np.asarray(ref)
    if test.size == 0:
        return np.nan
    return float(np.max(np.abs(test - ref)))


def norm_ratio(test, ref):
    test = np.asarray(test)
    ref = np.asarray(ref)
    test_norm = np.linalg.norm(test)
    ref_norm = np.linalg.norm(ref)
    return float(test_norm / ref_norm) if ref_norm != 0 else np.nan


def sanitize_overlap_mismatch(overlap, mismatch):
    """
    Avoid reporting tiny negative mismatch values caused by floating-point roundoff.
    This does not change rel_L2 or norm-ratio diagnostics.
    """
    overlap = float(overlap)
    mismatch = float(mismatch)

    if np.isfinite(overlap):
        overlap = min(1.0, max(-1.0, overlap))

    if np.isfinite(mismatch) and mismatch < 0.0 and abs(mismatch) < 1e-12:
        mismatch = 0.0

    return overlap, mismatch


def compare_wdm_arrays(test, ref):
    """
    Compare WDM coefficient arrays using project utilities plus simple diagnostics.
    """
    overlap, mismatch = wdm_mismatch(ref, test)
    overlap, mismatch = sanitize_overlap_mismatch(overlap, mismatch)

    return {
        "rel_l2": rel_l2(test, ref),
        "max_abs": max_abs(test, ref),
        "norm_ratio": norm_ratio(test, ref),
        "overlap": overlap,
        "mismatch": mismatch,
    }


def compare_td_arrays(test, ref):
    """
    Compare time-domain response arrays using project utility.
    """
    d = compare_response_signals(ref, test)
    overlap, mismatch = sanitize_overlap_mismatch(d["overlap"], d["mismatch"])

    return {
        "rel_l2": float(d["rel_L2"]),
        "max_abs": max_abs(test, ref),
        "norm_ratio": norm_ratio(test, ref),
        "overlap": overlap,
        "mismatch": mismatch,
    }


def compare_arrays(test, ref, domain="wdm"):
    """
    Generic comparison wrapper.

    domain="wdm" uses wdm_mismatch.
    domain="td" uses compare_response_signals.
    """
    if domain == "td":
        return compare_td_arrays(test, ref)
    return compare_wdm_arrays(test, ref)


def sync_result(x):
    """
    Force completion of JAX work before stopping timers.

    JAX execution can be asynchronous, so timing only the Python call can
    undercount runtime unless we block on the returned arrays.
    """
    try:
        leaves = jax.tree_util.tree_leaves(x)
        for y in leaves:
            if hasattr(y, "block_until_ready"):
                y.block_until_ready()
    except Exception:
        pass
    return x


def time_call(fn, warm=1, repeat=1, sync=True):
    warm = max(0, int(warm))
    repeat = max(1, int(repeat))

    for _ in range(warm):
        out = fn()
        if sync:
            sync_result(out)

    t0 = time.perf_counter()
    out = None

    for _ in range(repeat):
        out = fn()
        if sync:
            sync_result(out)

    t1 = time.perf_counter()
    return out, (t1 - t0) / repeat


def format_value(v):
    """
    Print helper only. Keep benchmark rows numeric internally.
    """
    if isinstance(v, (float, np.floating)):
        if not np.isfinite(v):
            return str(v)
        if abs(v) >= 1e4 or (abs(v) > 0 and abs(v) < 1e-3):
            return f"{v:.4e}"
        return f"{v:.6g}"
    return str(v)


def format_row(row):
    return {k: format_value(v) for k, v in row.items()}


def print_table(rows, columns, *, format_values=True):
    if format_values:
        rows_to_print = [format_row(row) for row in rows]
    else:
        rows_to_print = rows

    widths = [len(col) for col in columns]

    for row in rows_to_print:
        for i, col in enumerate(columns):
            widths[i] = max(widths[i], len(str(row.get(col, ""))))

    header = "  ".join(col.ljust(widths[i]) for i, col in enumerate(columns))
    print(header)
    print("-" * len(header))

    for row in rows_to_print:
        print("  ".join(str(row.get(col, "")).ljust(widths[i]) for i, col in enumerate(columns)))


print("Imported project utils and prepared comparison/timing helpers.")

Imported project utils and prepared comparison/timing helpers.


In [15]:
# --- Cell 4: direct target-mode time-shift benchmark / smoke test ---

direct_backends = selected_backends()

# Small synthetic WDM problem.
# This is only a correctness/smoke test for backend dispatch, not the full-year benchmark.
wdm = WDM.WDM.WDM_transform(
    dt=0.5,
    Nf=16,
    N=256,
    q=4,
    calc_m0=True,
)

rng = np.random.default_rng(1234)

w_xi = (
    rng.normal(size=(wdm.Nt, wdm.Nf))
    + 1j * rng.normal(size=(wdm.Nt, wdm.Nf))
).astype(np.complex128)

t_shift = 0.12 * np.sin(
    np.linspace(0.0, 2.0 * np.pi, wdm.Nt)
)

# For this tiny WDM grid, cap L_trunc so we do not use nonsensical large lag support.
SMOKE_L_TRUNC_MAX = min(3, wdm.Nt - 1)

results_direct = {}

for name, cfg in direct_backends.items():
    L_trunc = min(int(cfg["L_trunc"]), SMOKE_L_TRUNC_MAX)
    kwargs = dict(cfg["kwargs"])

    # For the tiny synthetic case, make sure lag_block_size is not larger than n_lags.
    n_lags = 2 * L_trunc + 1
    if "lag_block_size" in kwargs:
        kwargs["lag_block_size"] = min(int(kwargs["lag_block_size"]), n_lags)

    print(f"Running direct backend: {name} | smoke_L_trunc={L_trunc} | kwargs={kwargs}")

    def run(L_trunc=L_trunc, kwargs=kwargs):
        return tsf.wdm_time_shift_variable(
            wdm,
            w_xi,
            t_shift,
            Nf=wdm.Nf,
            L_trunc=L_trunc,
            **kwargs,
        )

    out, t_avg = time_call(run, warm=1, repeat=1)

    results_direct[name] = {
        "L_trunc": L_trunc,
        "n_lags": n_lags,
        "kwargs": kwargs,
        "time_s": float(t_avg),
        "output": out,
    }

# Use the baseline chunked backend as reference if available.
if "baseline_chunked_R1024_L25" in results_direct:
    ref_name = "baseline_chunked_R1024_L25"
else:
    ref_name = next(iter(results_direct))

ref_out = results_direct[ref_name]["output"]
ref_time = results_direct[ref_name]["time_s"]

rows_direct = []

for name, row in results_direct.items():
    cmp = compare_arrays(row["output"], ref_out, domain="wdm")
    speedup = ref_time / row["time_s"] if row["time_s"] > 0 else np.nan

    rows_direct.append({
        "backend": name,
        "L_trunc": int(row["L_trunc"]),
        "n_lags": int(row["n_lags"]),
        "time_s": float(row["time_s"]),
        "time_ms": 1e3 * float(row["time_s"]),
        "speedup_vs_ref": float(speedup),
        "rel_l2": float(cmp["rel_l2"]),
        "norm_ratio": float(cmp["norm_ratio"]),
        "mismatch": float(cmp["mismatch"]),
        "max_abs": float(cmp["max_abs"]),
    })

print()
print("Reference backend:", ref_name)
print_table(
    rows_direct,
    [
        "backend",
        "L_trunc",
        "n_lags",
        "time_ms",
        "speedup_vs_ref",
        "rel_l2",
        "norm_ratio",
        "mismatch",
        "max_abs",
    ],
)

Running direct backend: baseline_chunked_R1024_L25 | smoke_L_trunc=3 | kwargs={'assembly_backend': 'lagfirst_chunked', 'assembly_precision': 'complex64', 'row_chunk_size': 1024, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear'}
Running direct backend: lagblock_L16_R2048_B16 | smoke_L_trunc=3 | kwargs={'assembly_backend': 'lagfirst_chunked_lagblock', 'assembly_precision': 'complex64', 'row_chunk_size': 2048, 'lag_block_size': 7, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear'}
Running direct backend: lagblock_L20_R2048_B16 | smoke_L_trunc=3 | kwargs={'assembly_backend': 'lagfirst_chunked_lagblock', 'assembly_precision': 'complex64', 'row_chunk_size': 2048, 'lag_block_size': 7, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear'}
Running direct backend: lagblock_L25_R2048_B32 | smoke_L_trunc=3 | kwargs={'assembly_backend': 'lagfirst_chunked_lagblock', 'assembly_precision': 'complex64', 'row_chun

In [9]:
# --- Cell 5: generate waveform and orbit objects once ---

# Laptop-safe defaults.
# Increase these only after the smoke benchmark is stable.
DURATION_YR = 1.0
dt = 10
Nf = 100
q_wdm = 4
L_trunc_value = L_TRUNC_REF

alpha = 0.01
tukey_factor = 4

theta = np.pi / 2
phi = 0.0

print("Setting up waveform/orbits...")
print("DURATION_YR:", DURATION_YR)
print("dt:", dt)
print("Nf:", Nf)
print("q_wdm:", q_wdm)
print("L_trunc_value:", L_trunc_value)

td_gen = GenerateEMRIWaveform(
    "FastKerrEccentricEquatorialFlux",
    sum_kwargs=dict(pad_output=True, odd_len=True),
    return_list=True,
)

orbit_data = read_orbit_file("esa-trailing-orbits.h5")
orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
orbits_trailing = InterpolatedOrbits(
    orbit_times,
    orbit_data.positions,
    orbit_data.velocities,
)

t0 = orbit_data.t0 + 0.1 * YRSID_SI

t0 = orbit_data.t0 + 0.1 * YRSID_SI
T_obs = DURATION_YR * YRSID_SI
N = int(np.round(T_obs / dt))

model_kwargs = {
    "td_gen": td_gen,
    "m1": 1e6,
    "m2": 10.0,
    "a": 0.9,
    "p0": 8.5,
    "e0": 0.3,
    "x0": 1.0,
    "dist": 1.0,
    "theta": theta,
    "phi": phi,
    "qK": np.pi / 3,
    "phiK": np.pi / 3,
    "Phi_phi0": np.pi / 3,
    "Phi_theta0": 0.0,
    "Phi_r0": np.pi / 3,
    "eps": 1e-2,
}

window_kwargs = {
    "alpha": alpha,
    "tukey_factor": tukey_factor,
}

print("Generating waveform...")
print("N before padding:", N)

times_pad, hplus_td, hcross_td, pad_info = generate_windowed_padded_waveform(
    waveform_model="emri",
    t0=t0,
    dt=dt,
    N=N,
    model_kwargs=model_kwargs,
    window_function="tukey",
    window_kwargs=window_kwargs,
    wdm_Nf=Nf,
    return_padding_info=True,
)

print("Generated waveform.")
print("times_pad shape:", times_pad.shape)
print("hplus_td shape:", hplus_td.shape)
print("hcross_td shape:", hcross_td.shape)
print("pad_info:", pad_info)

Setting up waveform/orbits...
DURATION_YR: 1.0
dt: 10
Nf: 100
q_wdm: 4
L_trunc_value: 25
Generating waveform...
N before padding: 3155815
Generated waveform.
times_pad shape: (3282200,)
hplus_td shape: (3282200,)
hcross_td shape: (3282200,)
pad_info: {'original_length': 3155815, 'window_pad_left': 63117, 'window_pad_right': 63117, 'window_function': 'tukey', 'wdm_Nf': 100, 'wdm_tail_pad_right': 151, 'wdm_target_length': 3282200, 'wdm_Nt': 32822}


In [10]:
# --- Cell 5b: build TD one-arm reference and transform to WDM ---

print("Computing direct TD one-arm reference...")

TD_NUM_INT_POINTS = 80

def run_td_reference():
    return one_arm_response_td(
        times_pad,
        1,
        2,
        orbits_trailing,
        theta,
        phi,
        hplus_td,
        hcross_td,
        num_int_points=TD_NUM_INT_POINTS,
    )

td_ref_signal, td_ref_time_s = time_call(
    run_td_reference,
    warm=0,
    repeat=1,
)

print("TD reference shape:", td_ref_signal.shape)
print("TD reference runtime_s:", td_ref_time_s)

if not np.all(np.isfinite(td_ref_signal)):
    print("WARNING: TD reference contains non-finite values.")


# Build a WDM transform matching the benchmark configuration.
# This should match the WDM tiling used by one_arm_response_wdm_dT for already padded inputs.
wdm_td_ref = WDM.WDM.WDM_transform(
    dt=dt,
    Nf=Nf,
    N=len(times_pad),
    q=q_wdm,
    calc_m0=True,
    d=4,
    A_frac=0.25,
)

print("TD-reference WDM object:")
print("  Nt:", wdm_td_ref.Nt)
print("  Nf:", wdm_td_ref.Nf)
print("  N:", wdm_td_ref.N)
print("  dT:", wdm_td_ref.dT)

td_ref_wdm = wdm_td_ref.dwt(td_ref_signal)

print("TD reference WDM shape:", td_ref_wdm.shape)

# Conservative science-region cut:
# remove explicit window padding, Tukey taper-affected margin, WDM tail padding, and m=0.
cut_info = compute_wdm_cut_info(
    original_length=N,
    Nf=Nf,
    dt=dt,
    alpha=alpha,
    tukey_factor=tukey_factor,
    window_pad_left=pad_info.get("window_pad_left", None),
    window_pad_right=pad_info.get("window_pad_right", None),
    include_wdm_tail_padding=True,
    include_taper_margin=True,
    wdm_dT=Nf * dt,
    wdm_Nt=td_ref_wdm.shape[0],
    drop_m0=True,
)

td_ref_wdm_cut = apply_wdm_cut(td_ref_wdm, cut_info)

print("cut_info:")
print({
    "time_slice": cut_info["time_slice"],
    "freq_slice": cut_info["freq_slice"],
    "left_cut": cut_info["left_cut"],
    "right_cut": cut_info["right_cut"],
    "trim_m0": cut_info["trim_m0"],
})

print("TD reference WDM cut shape:", td_ref_wdm_cut.shape)

Computing direct TD one-arm reference...
TD reference shape: (3282200,)
TD reference runtime_s: 21.448359770001844
TD-reference WDM object:
  Nt: 32822
  Nf: 100
  N: 3282200
  dT: 1000.0
TD reference WDM shape: (32822, 100)
cut_info:
{'time_slice': slice(789, 32031, None), 'freq_slice': slice(1, None, None), 'left_cut': 789, 'right_cut': 791, 'trim_m0': True}
TD reference WDM cut shape: (31242, 99)


In [16]:
# --- Cell 6: one-arm WDM response benchmark against TD-derived WDM reference ---
# Context-reused version: builds WDM/DWT context once, then runs each backend.

one_arm_backends = selected_backends()

print("Running one-arm benchmark with backends:")
for name, cfg in one_arm_backends.items():
    print(f"  {name}: L_trunc={cfg['L_trunc']}, kwargs={cfg['kwargs']}")

if not one_arm_backends:
    raise RuntimeError("No active one-arm backends. Check selected_backends().")


# -------------------------------------------------------------------
# Build reusable WDM context once
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("Building reusable WDM context once...")

sky_basis = build_sky_basis(theta, phi)

def build_one_arm_context():
    return _build_wdm_dT_context(
        times_pad,
        hplus_td,
        hcross_td,
        Nf=Nf,
        d=4,
        q=q_wdm,
        calc_m0=True,
    )

one_arm_context, one_arm_context_time_s = time_call(
    build_one_arm_context,
    warm=0,
    repeat=1,
)

wdm_context = one_arm_context["wdm"]

print("context_build_time_s:", one_arm_context_time_s)
print("WDM context:")
print("  Nt:", wdm_context.Nt)
print("  Nf:", wdm_context.Nf)
print("  N:", wdm_context.N)
print("  dT:", wdm_context.dT)
print("  dF:", wdm_context.dF)


# -------------------------------------------------------------------
# Optional warm-up with the first backend
# -------------------------------------------------------------------

WARM_BACKEND_BEFORE_TIMING = False

if WARM_BACKEND_BEFORE_TIMING:
    first_name = next(iter(one_arm_backends))
    first_cfg = one_arm_backends[first_name]
    first_L = int(first_cfg["L_trunc"])
    first_kwargs = dict(first_cfg["kwargs"])

    print("\nWarming context-reused response path with backend:", first_name)
    print("warm L_trunc:", first_L)
    print("warm kwargs:", first_kwargs)

    warm_out = _one_arm_response_wdm_dT_from_context(
        one_arm_context,
        1,
        2,
        orbits_trailing,
        theta,
        phi,
        return_td=False,
        L_trunc=first_L,
        variable_shift_kwargs=first_kwargs,
        sky_basis=sky_basis,
    )
    sync_result(warm_out)

    print("Warm-up complete.")


# -------------------------------------------------------------------
# Run each backend using the same context
# -------------------------------------------------------------------

results_one_arm = {}

for name, cfg in one_arm_backends.items():
    L_trunc = int(cfg["L_trunc"])
    kwargs = dict(cfg["kwargs"])
    n_lags = n_lags_from_L(L_trunc)

    # Defensive: make sure lag_block_size is valid for this L_trunc.
    if "lag_block_size" in kwargs:
        kwargs["lag_block_size"] = min(int(kwargs["lag_block_size"]), n_lags)

    print("\n" + "=" * 80)
    print("Running one-arm backend:", name)
    print("L_trunc:", L_trunc)
    print("n_lags:", n_lags)
    print("kwargs:", kwargs)

    def run(L_trunc=L_trunc, kwargs=kwargs):
        return _one_arm_response_wdm_dT_from_context(
            one_arm_context,
            1,
            2,
            orbits_trailing,
            theta,
            phi,
            return_td=False,
            L_trunc=L_trunc,
            variable_shift_kwargs=kwargs,
            sky_basis=sky_basis,
        )

    out, t_avg = time_call(run, warm=1, repeat=1)

    if not np.all(np.isfinite(out)):
        print("WARNING: output contains non-finite values.")

    print("output shape:", out.shape)
    print("runtime_s_context_reused:", t_avg)

    results_one_arm[name] = {
        "L_trunc": L_trunc,
        "n_lags": n_lags,
        "kwargs": kwargs,
        "time_s": float(t_avg),
        "context_build_time_s": float(one_arm_context_time_s),
        "output": out,
    }


# -------------------------------------------------------------------
# Main reference: direct TD response transformed to WDM
# -------------------------------------------------------------------

if "td_ref_wdm_cut" not in globals():
    raise RuntimeError("td_ref_wdm_cut is not defined. Run Cell 5b first.")

if "cut_info" not in globals():
    raise RuntimeError("cut_info is not defined. Run Cell 5b first.")

ref_name = "td_reference_wdm"
ref_out_cut = td_ref_wdm_cut
ref_time = td_ref_time_s if "td_ref_time_s" in globals() else np.nan

print("\nMain reference:", ref_name)
print("TD WDM reference cut shape:", ref_out_cut.shape)


# -------------------------------------------------------------------
# Internal WDM reference for backend-to-backend diagnostics
# -------------------------------------------------------------------

# Prefer the L_trunc reference case if it exists.
preferred_internal_refs = [
    f"lagblock_L{L_TRUNC_REF}_R2048_B32",
    f"lagblock_L{L_TRUNC_REF}_R1024_B32",
    "baseline_chunked_R1024_L25",
]

faithful_name = None
for candidate in preferred_internal_refs:
    if candidate in results_one_arm:
        faithful_name = candidate
        break

# Fallback: use first available backend.
if faithful_name is None and results_one_arm:
    faithful_name = next(iter(results_one_arm))

faithful_cut = None
faithful_time = np.nan

if faithful_name is not None:
    faithful_full = results_one_arm[faithful_name]["output"]
    faithful_cut = apply_wdm_cut(faithful_full, cut_info)
    faithful_time = float(results_one_arm[faithful_name]["time_s"])

    print("Internal WDM reference:", faithful_name)
    print("Internal WDM reference full shape:", faithful_full.shape)
    print("Internal WDM reference cut shape:", faithful_cut.shape)

    if faithful_name != f"lagblock_L{L_TRUNC_REF}_R2048_B32":
        print(
            "NOTE: internal WDM reference is simply the selected reference backend. "
            "Use TD-derived WDM diagnostics as the main science comparison."
        )
else:
    print("No internal WDM reference found.")


# -------------------------------------------------------------------
# Build comparison table
# -------------------------------------------------------------------

rows_one_arm = []

for name, row in results_one_arm.items():
    out_full = row["output"]
    out_cut = apply_wdm_cut(out_full, cut_info)

    if out_cut.shape != ref_out_cut.shape:
        raise ValueError(
            f"Cut shape mismatch for {name}: "
            f"out_cut.shape={out_cut.shape}, ref_out_cut.shape={ref_out_cut.shape}"
        )

    # Main science comparison: WDM backend vs direct TD response transformed to WDM.
    cmp_td = compare_arrays(out_cut, ref_out_cut, domain="wdm")

    # Backend-only diagnostic: WDM backend vs chosen internal WDM route.
    if faithful_cut is not None:
        if out_cut.shape != faithful_cut.shape:
            raise ValueError(
                f"Faithful cut shape mismatch for {name}: "
                f"out_cut.shape={out_cut.shape}, faithful_cut.shape={faithful_cut.shape}"
            )

        cmp_faithful = compare_arrays(out_cut, faithful_cut, domain="wdm")
        rel_l2_vs_faithful = float(cmp_faithful["rel_l2"])
        norm_ratio_vs_faithful = float(cmp_faithful["norm_ratio"])
        overlap_vs_faithful = float(cmp_faithful["overlap"])
        mismatch_vs_faithful = float(cmp_faithful["mismatch"])
        max_abs_vs_faithful = float(cmp_faithful["max_abs"])
    else:
        rel_l2_vs_faithful = np.nan
        norm_ratio_vs_faithful = np.nan
        overlap_vs_faithful = np.nan
        mismatch_vs_faithful = np.nan
        max_abs_vs_faithful = np.nan

    speedup_vs_faithful = (
        faithful_time / row["time_s"]
        if np.isfinite(faithful_time) and row["time_s"] > 0
        else np.nan
    )

    estimated_one_shot_warm_s = one_arm_context_time_s + row["time_s"]

    kwargs = row["kwargs"]

    rows_one_arm.append({
        "backend": name,
        "L_trunc": int(row["L_trunc"]),
        "n_lags": int(row["n_lags"]),
        "row_chunk_size": int(kwargs.get("row_chunk_size", -1)) if "row_chunk_size" in kwargs else "",
        "lag_block_size": int(kwargs.get("lag_block_size", -1)) if "lag_block_size" in kwargs else "",
        "interp_points": int(kwargs.get("tl_tp_interp_points", -1)) if "tl_tp_interp_points" in kwargs else "",
        "time_s": float(row["time_s"]),
        "time_ms": 1e3 * float(row["time_s"]),
        "context_build_s": float(one_arm_context_time_s),
        "est_one_shot_warm_s": float(estimated_one_shot_warm_s),
        "speedup_vs_faithful": float(speedup_vs_faithful),
        "full_shape": str(out_full.shape),
        "cut_shape": str(out_cut.shape),

        # Main comparison against direct TD response transformed to WDM.
        "rel_l2_vs_td": float(cmp_td["rel_l2"]),
        "norm_ratio_vs_td": float(cmp_td["norm_ratio"]),
        "overlap_vs_td": float(cmp_td["overlap"]),
        "mismatch_vs_td": float(cmp_td["mismatch"]),
        "max_abs_vs_td": float(cmp_td["max_abs"]),

        # Backend-only comparison against selected internal WDM route.
        "rel_l2_vs_faithful": rel_l2_vs_faithful,
        "norm_ratio_vs_faithful": norm_ratio_vs_faithful,
        "overlap_vs_faithful": overlap_vs_faithful,
        "mismatch_vs_faithful": mismatch_vs_faithful,
        "max_abs_vs_faithful": max_abs_vs_faithful,
    })


print("\nOne-arm benchmark summary on WDM cut region:")
print_table(
    rows_one_arm,
    [
        "backend",
        "L_trunc",
        "n_lags",
        "row_chunk_size",
        "lag_block_size",
        "interp_points",
        "time_ms",
        "context_build_s",
        "est_one_shot_warm_s",
        "speedup_vs_faithful",
        "rel_l2_vs_td",
        "norm_ratio_vs_td",
        "mismatch_vs_td",
        "rel_l2_vs_faithful",
        "norm_ratio_vs_faithful",
        "mismatch_vs_faithful",
        "max_abs_vs_faithful",
    ],
)

Running one-arm benchmark with backends:
  baseline_chunked_R1024_L25: L_trunc=25, kwargs={'assembly_backend': 'lagfirst_chunked', 'assembly_precision': 'complex64', 'row_chunk_size': 1024, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear'}
  lagblock_L16_R2048_B16: L_trunc=16, kwargs={'assembly_backend': 'lagfirst_chunked_lagblock', 'assembly_precision': 'complex64', 'row_chunk_size': 2048, 'lag_block_size': 16, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear'}
  lagblock_L20_R2048_B16: L_trunc=20, kwargs={'assembly_backend': 'lagfirst_chunked_lagblock', 'assembly_precision': 'complex64', 'row_chunk_size': 2048, 'lag_block_size': 16, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear'}
  lagblock_L25_R2048_B32: L_trunc=25, kwargs={'assembly_backend': 'lagfirst_chunked_lagblock', 'assembly_precision': 'complex64', 'row_chunk_size': 2048, 'lag_block_size': 32, 'tl_tp_mode': 'interp', 'tl_tp_inter

## Optional TDI Benchmark (Heavy)

Set `RUN_TDI = True` to execute. This is intentionally disabled on laptops by default.

In [17]:
# --- Optional Cell: TD Gen-2 Michelson TDI reference on a safe short slice ---

RUN_TDI_TD_REFERENCE = False

# Full TD TDI reference is too memory-heavy at multi-million sample length.
# Use a short slice only for smoke/accuracy sanity checks.
TDI_TD_REFERENCE_MAX_SAMPLES = 100_000

# Laptop-safe defaults. Increase only after this is stable.
TDI_TD_NUM_INT_POINTS = 40
TDI_TD_ORDER = 7
TDI_TD_MODE = "zero"

if RUN_TDI_TD_REFERENCE:
    print("Computing TD Gen-2 Michelson TDI reference on a short slice...")
    print("TDI_TD_REFERENCE_MAX_SAMPLES:", TDI_TD_REFERENCE_MAX_SAMPLES)
    print("TDI_TD_NUM_INT_POINTS:", TDI_TD_NUM_INT_POINTS)
    print("TDI_TD_ORDER:", TDI_TD_ORDER)
    print("TDI_TD_MODE:", TDI_TD_MODE)

    # Make short slice compatible with WDM tiling.
    block = int(2 * Nf)
    n_tdi_ref_raw = min(int(TDI_TD_REFERENCE_MAX_SAMPLES), len(times_pad))
    n_tdi_ref = max(block, (n_tdi_ref_raw // block) * block)

    if n_tdi_ref > len(times_pad):
        raise RuntimeError(
            f"Computed n_tdi_ref={n_tdi_ref} exceeds len(times_pad)={len(times_pad)}."
        )

    times_tdi_td = times_pad[:n_tdi_ref]
    hplus_tdi_td = hplus_td[:n_tdi_ref]
    hcross_tdi_td = hcross_td[:n_tdi_ref]

    print("TD TDI slice length:", n_tdi_ref)
    print("TD TDI slice duration days:", (times_tdi_td[-1] - times_tdi_td[0]) / 86400.0)

    tdi_td_shifts_cache = {}

    def run_tdi_td_reference():
        return gen2_michelson_td(
            times_tdi_td,
            hplus_tdi_td,
            hcross_tdi_td,
            theta,
            phi,
            orbits_trailing,
            order=TDI_TD_ORDER,
            mode=TDI_TD_MODE,
            num_int_points=TDI_TD_NUM_INT_POINTS,
            shifts_cache=tdi_td_shifts_cache,
        )

    (X_td, Y_td, Z_td), tdi_td_time_s = time_call(
        run_tdi_td_reference,
        warm=0,
        repeat=1,
    )

    print("TD TDI runtime_s:", tdi_td_time_s)
    print("TD X/Y/Z shapes:", X_td.shape, Y_td.shape, Z_td.shape)

    if not (
        np.all(np.isfinite(X_td))
        and np.all(np.isfinite(Y_td))
        and np.all(np.isfinite(Z_td))
    ):
        print("WARNING: TD TDI reference contains non-finite values.")

    wdm_tdi_ref = WDM.WDM.WDM_transform(
        dt=dt,
        Nf=Nf,
        N=len(times_tdi_td),
        q=q_wdm,
        calc_m0=True,
        d=4,
        A_frac=0.25,
    )

    print("TDI TD-reference WDM object:")
    print("  Nt:", wdm_tdi_ref.Nt)
    print("  Nf:", wdm_tdi_ref.Nf)
    print("  N:", wdm_tdi_ref.N)

    X_td_wdm = wdm_tdi_ref.dwt(X_td)
    Y_td_wdm = wdm_tdi_ref.dwt(Y_td)
    Z_td_wdm = wdm_tdi_ref.dwt(Z_td)

    print("TD TDI WDM shapes:", X_td_wdm.shape, Y_td_wdm.shape, Z_td_wdm.shape)

    # For the short slice, compare on whole WDM grid except m=0.
    cut_info_tdi = {
        "time_slice": slice(0, X_td_wdm.shape[0]),
        "freq_slice": slice(1, X_td_wdm.shape[1]),
        "left_cut": 0,
        "right_cut": 0,
        "trim_m0": True,
    }

    X_td_wdm_cut = apply_wdm_cut(X_td_wdm, cut_info_tdi)
    Y_td_wdm_cut = apply_wdm_cut(Y_td_wdm, cut_info_tdi)
    Z_td_wdm_cut = apply_wdm_cut(Z_td_wdm, cut_info_tdi)

    tdi_td_ref_concat_cut = np.concatenate([
        np.asarray(X_td_wdm_cut).ravel(),
        np.asarray(Y_td_wdm_cut).ravel(),
        np.asarray(Z_td_wdm_cut).ravel(),
    ])

    print("cut_info_tdi:")
    print({
        "time_slice": cut_info_tdi["time_slice"],
        "freq_slice": cut_info_tdi["freq_slice"],
        "left_cut": cut_info_tdi["left_cut"],
        "right_cut": cut_info_tdi["right_cut"],
        "trim_m0": cut_info_tdi["trim_m0"],
    })

    print("TD TDI WDM cut shapes:", X_td_wdm_cut.shape, Y_td_wdm_cut.shape, Z_td_wdm_cut.shape)

else:
    print("RUN_TDI_TD_REFERENCE is False; skipping TD TDI reference.")

RUN_TDI_TD_REFERENCE is False; skipping TD TDI reference.


In [18]:
# --- Optional Cell: profiled split-staged full-size Gen-2 Michelson TDI WDM benchmark ---

from WDM_LISAresponse.code.lisa_response.TDI import (
    precompute_gen2_michelson_wdm_geometry,
    precompute_gen2_michelson_wdm_waveform,
    gen2_michelson_wdm_from_waveform_precomputed,
)

RUN_TDI = True

# Timing controls.
A0_GEOM_WARM_RUNS = 0
A1_WAVE_WARM_RUNS = 0
B_TDI_WARM_RUNS = 0

A0_GEOM_REPEAT = 1
A1_WAVE_REPEAT = 1
B_TDI_REPEAT = 1

# Profiling controls.
PROFILE_SHIFT_STAGES = True
RETURN_SHIFT_PROFILE = True

rows_tdi = []
results_tdi = {}

if RUN_TDI:
    print("Running PROFILED split-staged full-size Gen-2 Michelson TDI WDM benchmark")
    print("Nf:", Nf)
    print("q_wdm:", q_wdm)
    print("PROFILE_SHIFT_STAGES:", PROFILE_SHIFT_STAGES)
    print("RETURN_SHIFT_PROFILE:", RETURN_SHIFT_PROFILE)

    print("\nTDI candidates:")
    for name, cfg in TDI_CANDIDATES.items():
        print(f"  {name}:")
        print(f"    L_trunc: {cfg['L_trunc']}")
        print(f"    tdi_assembly_mode: {cfg.get('tdi_assembly_mode', 'batch_map')}")
        print(f"    accumulator_chunk: {cfg.get('accumulator_chunk', 1)}")
        print(f"    one_arm_kwargs: {cfg['one_arm_kwargs']}")
        print(f"    downstream_kwargs: {cfg['downstream_kwargs']}")

    staged_cache = {}

    for cand_name, cand in TDI_CANDIDATES.items():
        L_cand = int(cand["L_trunc"])
        one_arm_kwargs = dict(cand["one_arm_kwargs"])
        downstream_kwargs = dict(cand["downstream_kwargs"])

        tdi_assembly_mode = str(cand.get("tdi_assembly_mode", "batch_map"))
        accumulator_chunk = int(cand.get("accumulator_chunk", 1))

        precompute_key = (
            L_cand,
            tuple(sorted((str(k), repr(v)) for k, v in one_arm_kwargs.items())),
        )

        print("\n" + "#" * 100)
        print("TDI candidate:", cand_name)
        print("L_trunc:", L_cand)
        print("tdi_assembly_mode:", tdi_assembly_mode)
        print("accumulator_chunk:", accumulator_chunk)
        print("one_arm_kwargs:", one_arm_kwargs)
        print("downstream_kwargs:", downstream_kwargs)

        if precompute_key in staged_cache:
            print("\nReusing cached A0/A1 staged precompute for this dense setting.")

            cached = staged_cache[precompute_key]

            tdi_geom_precomp = cached["tdi_geom_precomp"]
            tdi_wave_precomp = cached["tdi_wave_precomp"]

            tdi_geom_time_s = cached["geometry_precompute_s"]
            tdi_wave_time_s = cached["waveform_precompute_s"]

            a1_context_s = cached["waveform_context_build_s"]
            a1_links_s = cached["six_link_compute_s"]
            a1_jobs_s = cached["job_build_s"]
            a1_outer_minus_metadata_s = cached["a1_outer_minus_metadata_s"]

            wdm_tdi = tdi_geom_precomp["wdm"]
            wdm_wave = tdi_wave_precomp["wdm"]
            n_tdi_jobs = int(cached["n_tdi_jobs"])

        else:
            # --------------------------------------------------------------
            # A0: geometry/static precompute
            # --------------------------------------------------------------

            print("\n" + "=" * 80)
            print("A0: precomputing Gen-2 Michelson WDM geometry/static inputs...")

            def run_tdi_geometry_precompute(
                L_cand=L_cand,
                one_arm_kwargs=one_arm_kwargs,
            ):
                return precompute_gen2_michelson_wdm_geometry(
                    times_pad,
                    theta,
                    phi,
                    orbits_trailing,
                    Nf=Nf,
                    L_trunc=L_cand,
                    q=q_wdm,
                    d=4,
                    variable_shift_kwargs=one_arm_kwargs,
                )

            tdi_geom_precomp, tdi_geom_time_s = time_call(
                run_tdi_geometry_precompute,
                warm=A0_GEOM_WARM_RUNS,
                repeat=A0_GEOM_REPEAT,
            )

            print("A0 geometry_precompute_s:", tdi_geom_time_s)

            if not isinstance(tdi_geom_precomp, dict):
                raise TypeError(
                    f"Expected tdi_geom_precomp to be dict, got {type(tdi_geom_precomp)}"
                )

            if "wdm" not in tdi_geom_precomp:
                raise RuntimeError(
                    "geom_precomp does not contain 'wdm'; A1 cannot reuse geometry WDM."
                )

            wdm_tdi = tdi_geom_precomp["wdm"]
            print("geom WDM:")
            print("  Nt:", wdm_tdi.Nt)
            print("  Nf:", wdm_tdi.Nf)
            print("  N:", wdm_tdi.N)
            print("  dT:", wdm_tdi.dT)
            print("  dF:", wdm_tdi.dF)

            forbidden_keys = [
                "hplus",
                "hcross",
                "waveform_plus",
                "waveform_cross",
                "link_w",
                "x_jobs",
                "y_jobs",
                "z_jobs",
                "all_jobs",
            ]
            for forbidden_key in forbidden_keys:
                if forbidden_key in tdi_geom_precomp:
                    print(
                        f"WARNING: geometry precomp unexpectedly contains waveform/job key: "
                        f"{forbidden_key}"
                    )

            # --------------------------------------------------------------
            # A1: waveform-specific precompute
            # --------------------------------------------------------------

            print("\n" + "=" * 80)
            print("A1: precomputing waveform-specific Gen-2 Michelson WDM inputs...")

            def run_tdi_waveform_precompute(
                tdi_geom_precomp=tdi_geom_precomp,
                one_arm_kwargs=one_arm_kwargs,
            ):
                return precompute_gen2_michelson_wdm_waveform(
                    tdi_geom_precomp,
                    hplus_td,
                    hcross_td,
                    variable_shift_kwargs=one_arm_kwargs,
                )

            tdi_wave_precomp, tdi_wave_time_s = time_call(
                run_tdi_waveform_precompute,
                warm=A1_WAVE_WARM_RUNS,
                repeat=A1_WAVE_REPEAT,
            )

            print("A1 waveform_precompute_s:", tdi_wave_time_s)

            if not isinstance(tdi_wave_precomp, dict):
                raise TypeError(
                    f"Expected tdi_wave_precomp to be dict, got {type(tdi_wave_precomp)}"
                )

            a1_context_s = float(tdi_wave_precomp.get("waveform_context_build_s", np.nan))
            a1_links_s = float(tdi_wave_precomp.get("six_link_compute_s", np.nan))
            a1_jobs_s = float(tdi_wave_precomp.get("job_build_s", np.nan))
            a1_meta_sum_s = a1_context_s + a1_links_s + a1_jobs_s
            a1_outer_minus_metadata_s = float(tdi_wave_time_s - a1_meta_sum_s)

            print("\nA1 timing metadata:")
            print("  waveform_context_build_s:", a1_context_s)
            print("  six_link_compute_s:", a1_links_s)
            print("  job_build_s:", a1_jobs_s)
            print("  metadata_sum_s:", a1_meta_sum_s)
            print("  outer_A1_time_s:", tdi_wave_time_s)
            print("  outer_minus_metadata_s:", a1_outer_minus_metadata_s)

            if "wdm" not in tdi_wave_precomp:
                raise RuntimeError("wave_precomp does not contain 'wdm'.")

            wdm_wave = tdi_wave_precomp["wdm"]
            print("\nA1 WDM reuse check:")
            print("  wave_precomp['wdm'] is geom_precomp['wdm']:", wdm_wave is wdm_tdi)

            if "link_w" in tdi_wave_precomp:
                print("wave link_w keys:", sorted(tdi_wave_precomp["link_w"].keys()))

            n_tdi_jobs = len(tdi_wave_precomp.get("all_jobs", []))
            print("number of downstream TDI jobs:", n_tdi_jobs)

            staged_cache[precompute_key] = {
                "tdi_geom_precomp": tdi_geom_precomp,
                "tdi_wave_precomp": tdi_wave_precomp,
                "geometry_precompute_s": float(tdi_geom_time_s),
                "waveform_precompute_s": float(tdi_wave_time_s),
                "waveform_context_build_s": float(a1_context_s),
                "six_link_compute_s": float(a1_links_s),
                "job_build_s": float(a1_jobs_s),
                "a1_outer_minus_metadata_s": float(a1_outer_minus_metadata_s),
                "n_tdi_jobs": int(n_tdi_jobs),
            }

        # --------------------------------------------------------------
        # B: downstream TDI with profiling
        # --------------------------------------------------------------

        print("\n" + "=" * 80)
        print("B: running downstream staged TDI with shift profiling...")
        print("downstream kwargs:", downstream_kwargs)
        print("tdi_assembly_mode:", tdi_assembly_mode)
        print("accumulator_chunk:", accumulator_chunk)

        def run_downstream_tdi(
            tdi_wave_precomp=tdi_wave_precomp,
            downstream_kwargs=downstream_kwargs,
            L_cand=L_cand,
            tdi_assembly_mode=tdi_assembly_mode,
            accumulator_chunk=accumulator_chunk,
            cand_name=cand_name,
        ):
            return gen2_michelson_wdm_from_waveform_precomputed(
                tdi_wave_precomp,
                variable_shift_kwargs=downstream_kwargs,
                L_trunc=L_cand,
                Nf=Nf,
                tdi_assembly_mode=tdi_assembly_mode,
                accumulator_chunk=accumulator_chunk,
                profile_shift_stages=PROFILE_SHIFT_STAGES,
                profile_label=cand_name,
                return_shift_profile=RETURN_SHIFT_PROFILE,
            )

        profiled_out, tdi_downstream_time_s = time_call(
            run_downstream_tdi,
            warm=B_TDI_WARM_RUNS,
            repeat=B_TDI_REPEAT,
        )

        if RETURN_SHIFT_PROFILE:
            out, shift_profile = profiled_out
        else:
            out = profiled_out
            shift_profile = {}

        X_wdm, Y_wdm, Z_wdm = out

        total_first_waveform_s = (
            tdi_geom_time_s
            + tdi_wave_time_s
            + tdi_downstream_time_s
        )
        total_next_waveform_s = (
            tdi_wave_time_s
            + tdi_downstream_time_s
        )

        finite = (
            np.all(np.isfinite(X_wdm))
            and np.all(np.isfinite(Y_wdm))
            and np.all(np.isfinite(Z_wdm))
        )

        if not finite:
            print("WARNING: TDI WDM output contains non-finite values.")

        print("X/Y/Z shapes:", X_wdm.shape, Y_wdm.shape, Z_wdm.shape)
        print("B downstream_tdi_s:", tdi_downstream_time_s)
        print("total_first_waveform_s:", total_first_waveform_s)
        print("total_next_waveform_s:", total_next_waveform_s)

        print("\nShift profile:")
        for k, v in shift_profile.items():
            print(f"  {k}: {v}")

        total_profile_s = float(shift_profile.get("total_s", np.nan))
        tl_tp_s = float(shift_profile.get("tl_tp_prepare_s", np.nan))
        assembly_s = float(shift_profile.get("jax_assembly_s", np.nan))
        post_s = float(shift_profile.get("postprocess_s", np.nan))
        other_s = float(shift_profile.get("other_s", np.nan))

        print("\nProfile fractions:")
        print(
            "  tl_tp_prepare_frac:",
            tl_tp_s / total_profile_s if total_profile_s > 0 else np.nan,
        )
        print(
            "  jax_assembly_frac:",
            assembly_s / total_profile_s if total_profile_s > 0 else np.nan,
        )
        print(
            "  postprocess_frac:",
            post_s / total_profile_s if total_profile_s > 0 else np.nan,
        )
        print(
            "  other_frac:",
            other_s / total_profile_s if total_profile_s > 0 else np.nan,
        )

        results_tdi[cand_name] = {
            "candidate": cand_name,
            "L_trunc": L_cand,
            "n_lags": n_lags_from_L(L_cand),
            "tdi_assembly_mode": tdi_assembly_mode,
            "accumulator_chunk": accumulator_chunk,
            "geometry_precompute_s": float(tdi_geom_time_s),
            "waveform_precompute_s": float(tdi_wave_time_s),
            "waveform_context_build_s": float(a1_context_s),
            "six_link_compute_s": float(a1_links_s),
            "job_build_s": float(a1_jobs_s),
            "a1_outer_minus_metadata_s": float(a1_outer_minus_metadata_s),
            "downstream_tdi_s": float(tdi_downstream_time_s),
            "total_first_waveform_s": float(total_first_waveform_s),
            "total_next_waveform_s": float(total_next_waveform_s),
            "n_tdi_jobs": int(n_tdi_jobs),
            "output": out,
            "finite": bool(finite),
            "one_arm_kwargs": dict(one_arm_kwargs),
            "downstream_kwargs": dict(downstream_kwargs),
            "shift_profile": dict(shift_profile),
        }

    # ------------------------------------------------------------------
    # Build compact profile table
    # ------------------------------------------------------------------

    for name, row in results_tdi.items():
        profile = row.get("shift_profile", {})

        total_profile_s = float(profile.get("total_s", np.nan))
        tl_tp_s = float(profile.get("tl_tp_prepare_s", np.nan))
        assembly_s = float(profile.get("jax_assembly_s", np.nan))
        post_s = float(profile.get("postprocess_s", np.nan))
        other_s = float(profile.get("other_s", np.nan))

        n_jobs = int(row["n_tdi_jobs"])
        B_per_job_ms = (
            1e3 * row["downstream_tdi_s"] / n_jobs
            if n_jobs > 0
            else np.nan
        )

        downstream_kwargs = row["downstream_kwargs"]

        rows_tdi.append({
            "candidate": name,
            "L_trunc": int(row["L_trunc"]),
            "n_lags": int(row["n_lags"]),
            "tdi_assembly_mode": str(row["tdi_assembly_mode"]),
            "accumulator_chunk": int(row["accumulator_chunk"]),
            "row_chunk_size": int(downstream_kwargs.get("row_chunk_size", -1)),
            "lag_block_size": int(downstream_kwargs.get("lag_block_size", -1)),
            "batch_chunk": int(downstream_kwargs.get("batch_chunk", -1)),
            "interp_points": int(downstream_kwargs.get("tl_tp_interp_points", -1)),

            "waveform_precompute_s": float(row["waveform_precompute_s"]),
            "six_link_compute_s": float(row["six_link_compute_s"]),
            "downstream_tdi_s": float(row["downstream_tdi_s"]),
            "B_per_job_ms": float(B_per_job_ms),
            "total_next_waveform_s": float(row["total_next_waveform_s"]),

            "profile_total_s": total_profile_s,
            "tl_tp_prepare_s": tl_tp_s,
            "jax_assembly_s": assembly_s,
            "postprocess_s": post_s,
            "other_s": other_s,

            "tl_tp_prepare_frac": tl_tp_s / total_profile_s if total_profile_s > 0 else np.nan,
            "jax_assembly_frac": assembly_s / total_profile_s if total_profile_s > 0 else np.nan,
            "postprocess_frac": post_s / total_profile_s if total_profile_s > 0 else np.nan,
            "other_frac": other_s / total_profile_s if total_profile_s > 0 else np.nan,

            "profile_n_jobs": int(profile.get("n_jobs", -1)),
            "finite": bool(row["finite"]),
        })

    print("\nPROFILED SPLIT-STAGED TDI WDM summary:")
    print_table(
        rows_tdi,
        [
            "candidate",
            "L_trunc",
            "n_lags",
            "tdi_assembly_mode",
            "row_chunk_size",
            "lag_block_size",
            "batch_chunk",
            "interp_points",
            "waveform_precompute_s",
            "six_link_compute_s",
            "downstream_tdi_s",
            "B_per_job_ms",
            "total_next_waveform_s",
            "profile_total_s",
            "tl_tp_prepare_s",
            "jax_assembly_s",
            "postprocess_s",
            "other_s",
            "tl_tp_prepare_frac",
            "jax_assembly_frac",
            "postprocess_frac",
            "other_frac",
            "profile_n_jobs",
            "finite",
        ],
    )

else:
    print("RUN_TDI is False; skipping split-staged TDI WDM benchmark.")

Running PROFILED split-staged full-size Gen-2 Michelson TDI WDM benchmark
Nf: 100
q_wdm: 4
PROFILE_SHIFT_STAGES: True
RETURN_SHIFT_PROFILE: True

TDI candidates:
  tdi_ref_L25_batch_map:
    L_trunc: 25
    tdi_assembly_mode: batch_map
    accumulator_chunk: 1
    one_arm_kwargs: {'assembly_backend': 'lagfirst_chunked_lagblock', 'assembly_precision': 'complex64', 'row_chunk_size': 2048, 'lag_block_size': 32, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear'}
    downstream_kwargs: {'assembly_backend': 'lagfirst_chunked_lagblock', 'assembly_precision': 'complex64', 'row_chunk_size': 2048, 'lag_block_size': 32, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear', 'batch_chunk': 1}
  tdi_safe_L20_batch_map:
    L_trunc: 20
    tdi_assembly_mode: batch_map
    accumulator_chunk: 1
    one_arm_kwargs: {'assembly_backend': 'lagfirst_chunked_lagblock', 'assembly_precision': 'complex64', 'row_chunk_size': 2048, 'lag_block_size': 16, '